In [1]:
# ========== 导入库和设置 ==========
# 本实验对推荐系统的各个模块进行消融研究，分析不同组件的贡献

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 设置中文字体（如果需要）
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置图片显示参数
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100

# 路径设置
TMP_DIR = Path('/workspace/recsys/tmp')

# 评价指标权重（统一指标计算用）
W_TAG = 0.5
W_DESC = 0.3  # Description维度（或Org维度）
W_CREATOR = 0.2

print('✅ 库导入成功')
print(f'TMP_DIR: {TMP_DIR}')

✅ 库导入成功
TMP_DIR: /workspace/recsys/tmp


In [2]:
# ========== 工具函数 ==========
# 定义可复用的工具函数

def add_unified_cols(df):
    """
    为DataFrame添加5个统一指标列
    
    统一指标公式:
        Unified@X = W_TAG * Tag-X + W_DESC * (Desc-X or Org-X) + W_CREATOR * Creator-X
    
    Args:
        df: 包含三维度评测指标的DataFrame
    
    Returns:
        df: 添加了Unified列的DataFrame（原地修改）
    """
    metrics = ['nDCG@20', 'MAP@20', 'MRR@20', 'P@20', 'R@20']
    
    for metric in metrics:
        tag_col = f'Tag-{metric}'
        # 优先使用Desc维度，如果没有则使用Org维度
        desc_col = f'Desc-{metric}' if f'Desc-{metric}' in df.columns else f'Org-{metric}'
        creator_col = f'Creator-{metric}'
        unified_col = f'Unified@{metric}'
        
        # 计算统一指标（缺失值用0填充）
        df[unified_col] = (
            W_TAG * df[tag_col].fillna(0) +
            W_DESC * df[desc_col].fillna(0) +
            W_CREATOR * df[creator_col].fillna(0)
        )
    
    return df

def plot_comparison(df, methods, title, ylabel='Score', figsize=(10, 6), 
                   my_method=None, show_values=True):
    """
    绘制并列柱状图比较不同方法
    
    Args:
        df: 包含Unified@指标的DataFrame
        methods: 要比较的方法列表
        title: 图表标题
        ylabel: y轴标签
        figsize: 图表大小
        my_method: 标记为"my method"的方法名称
        show_values: 是否在柱子上显示数值
    """
    # 筛选数据
    plot_df = df[df['method'].isin(methods)].copy()
    
    # 按methods顺序排序
    plot_df['__order'] = plot_df['method'].apply(lambda x: methods.index(x))
    plot_df = plot_df.sort_values('__order').drop(columns='__order')
    
    # 提取数据
    series_names = ['Unified@nDCG@20', 'Unified@MAP@20', 'Unified@MRR@20', 
                   'Unified@P@20', 'Unified@R@20']
    display_names = ['nDCG', 'MAP', 'MRR', 'P', 'R']
    
    M = len(series_names)  # 5个指标
    K = len(plot_df)       # 方法数
    
    # 创建图表
    fig, ax = plt.subplots(figsize=figsize)
    
    x = np.arange(M)
    width = 0.75 / K if K > 0 else 0.15
    
    # 绘制每个方法的柱子
    for i, (idx, row) in enumerate(plot_df.iterrows()):
        method_name = row['method']
        offset = (i - (K - 1) / 2.0) * width
        values = [row[col] for col in series_names]
        
        # 标记my method
        label = method_name
        if my_method and method_name == my_method:
            label = f'{method_name} (my method)'
        
        bars = ax.bar(x + offset, values, width=width, label=label)
        
        # 在柱子上标注数值
        if show_values:
            for bar in bars:
                height = bar.get_height()
                ax.annotate(f'{height:.3f}',
                           xy=(bar.get_x() + bar.get_width() / 2, height),
                           xytext=(0, 3),
                           textcoords='offset points',
                           ha='center', va='bottom',
                           fontsize=8)
    
    # 设置标签
    ax.set_xticks(x)
    ax.set_xticklabels(display_names)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    
    # 图例（加粗my method）
    leg = ax.legend(loc='best', frameon=True)
    if my_method:
        for text in leg.get_texts():
            if my_method in text.get_text():
                text.set_fontweight('bold')
    
    # 自动调整y轴范围（留点margin）
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax * 1.1)
    
    plt.tight_layout()
    plt.show()

print('✅ 工具函数定义完成')

✅ 工具函数定义完成


In [3]:
# ========== 第1步：模块消融（RR + Blend） ==========
# 目标：比较"三视图融合但没有RR+Blend的Fused3-RA"和"带RR+blend的Fused3-Blend-eta0.30（my method）"的差异

print('[第1步] 模块消融：RR + Blend 的贡献\n')

# 1. 读取数据
print('[1/4] 读取metrics数据...')
df_main = pd.read_csv(TMP_DIR / 'metrics_main.csv')
df_blend = pd.read_csv(TMP_DIR / 'metrics_fused3_blend_eta.csv')

# 2. 提取需要对比的方法
print('[2/4] 提取对比方法...')
row_fused3_ra = df_main[df_main['method'] == 'Fused3-RA'].copy()
row_fused3_blend = df_blend[df_blend['method'] == 'Fused3-Blend-eta0.30'].copy()

# 检查是否找到数据
if len(row_fused3_ra) == 0:
    print('⚠️  未找到 Fused3-RA，可用方法:', df_main['method'].tolist())
if len(row_fused3_blend) == 0:
    print('⚠️  未找到 Fused3-Blend-eta0.30，可用方法:', df_blend['method'].tolist())

# 合并数据
df_step1 = pd.concat([row_fused3_ra, row_fused3_blend], ignore_index=True)

# 3. 计算统一指标
print('[3/4] 计算统一指标...')
df_step1 = add_unified_cols(df_step1)

# 显示结果
print('\n统一指标结果:')
display_cols = ['method', 'Unified@nDCG@20', 'Unified@MAP@20', 'Unified@MRR@20', 
                'Unified@P@20', 'Unified@R@20']
print(df_step1[display_cols].to_string(index=False))

# 4. 绘制对比图
print('\n[4/4] 绘制对比图...')
methods = ['Fused3-RA', 'Fused3-Blend-eta0.30']
plot_comparison(
    df=df_step1,
    methods=methods,
    title='Step 1: Module Ablation - RR + Blend Contribution\n' +
          'Comparing Fused3-RA (w/o RR+Blend) vs Fused3-Blend-eta0.30 (w/ RR+Blend, my method)',
    ylabel='Unified Score',
    figsize=(12, 6),
    my_method='Fused3-Blend-eta0.30',
    show_values=True
)

print('\n✅ 第1步完成！')

[第1步] 模块消融：RR + Blend 的贡献

[1/4] 读取metrics数据...


FileNotFoundError: [Errno 2] No such file or directory: '/workspace/recsys/tmp/metrics_fused3_blend_eta.csv'